# An active wave beside a controlled boundary

**Research question:** how do measured voltage and estimated tissue contribution change as a finite electrode grid crosses an active viable–lesion boundary?

We start with a homogeneous, full-thickness slab. We pace the left edge to produce a wave along x and move the catheter across y. The intact and lesion conditions share the mesh coordinates, pacing schedule and electrode positions. The lesion is near-insulating tissue below y = 0; it is not a passive ionic model.

This notebook is the experiment's reading order. Each step calls an ordinary Python function in the accompanying files. The terminal runner calls the same functions. Array dimensions, units and the more specialised numerical operations are explained in their docstrings.

**Execution status:** this rewritten notebook has no executed cells. The rewrite has only been reviewed statically; simulations and tests must be run on your simulation computer if desired.


## 1. Load the experiment functions

Open this notebook from `3D_Code`, or start Jupyter from the project folder containing `3D_Code` and `Shared_Pipeline`. Use the Python environment that has NumPy, SciPy, pandas and Matplotlib installed. The simulator and mesher must be available separately.


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

codeFolder = Path.cwd()
if codeFolder.name != "3D_Code":
    codeFolder = codeFolder / "3D_Code"
if not (codeFolder / "boundary_setup.py").is_file():
    raise FileNotFoundError("Start this notebook in 3D_Code or its parent project folder")
sys.path.insert(0, str(codeFolder))

from boundary_setup import (
    loadConfig, validateConfig, prepareExperiment, simulateExperiment,
    readNodePositions, startAnalysis, checkCompletedSimulations,
)
from boundary_geometry import makeElectrodeLayout
from boundary_checks import checkExperiment
from boundary_analysis import (
    calibrateGain, processCondition, computeContributions, saveExperimentResults,
)
from boundary_figures import makeFigures


## 2. Choose the run and experimental settings

The JSON file contains all defaults; `3d_pilot.par` contains the CRN model, conductivities and solver settings. The assignments below are convenient notebook overrides. They match the current defaults, but edits here apply only to this notebook run. For terminal runs, edit a copy of the JSON file and pass it with `--config`.

Use a **new output folder** when changing geometry, electrodes, pacing, integration settings, the parameter template or the code that generates simulation inputs. Changing only analysis settings can reuse the raw simulations. The exact configuration and source are saved with the run.


In [ ]:
config = loadConfig(codeFolder / "configs" / "boundary_pilot.json")
runFolder = codeFolder / "runs" / "boundaryPilot"
processCount = 1  # Start serially. For >1, first set MPIEXEC_BIN to the matching launcher.

config["slabSizeMm"] = [40.0, 40.0, 4.0]
config["meshSpacingMm"] = 0.5
config["pacing"]["beatCount"] = 8
config["pacing"]["cycleLengthMs"] = 500.0
config["boundary"] = {"shape": "straight", "positionMm": 0.0}

validateConfig(config)
print(json.dumps(config, indent=2))


## 3. Look at the catheter before simulating

The grid has sixteen electrodes. Each electrode is a cylinder aligned with x. We average equally weighted samples on its lateral surface; the default is two axial positions and four angles. `gap` means the distance from the tissue surface to the nearest cylinder surface, not the cylinder centre.

Here we draw the centre-y = 0 placement at the first gap. Other placements translate the same grid across the boundary. No mesh or simulation is created in this step.


In [ ]:
layout = makeElectrodeLayout(config)
placements = layout["placements"]  # columns: centre x, centre y, surface gap (mm)
placementIndex = np.flatnonzero(
    np.isclose(placements[:, 1], 0) & np.isclose(placements[:, 2], config["electrodes"]["gapsMm"][0])
)[0]
centres = layout["centres"][placementIndex]  # (16 electrodes, xyz)
samples = layout["samples"][placementIndex]  # (16 electrodes, surface sample, xyz)

figure, axis = plt.subplots(figsize=(6, 6))
axis.scatter(samples[..., 0], samples[..., 1], s=10, label="Surface samples")
axis.scatter(centres[:, 0], centres[:, 1], marker="x", color="black", label="Centres")
for electrodeIndex, position in enumerate(centres):
    axis.annotate(f"E{electrodeIndex}", position[:2], xytext=(5, 5), textcoords="offset points")
axis.axhline(config["boundary"]["positionMm"], color="black", linestyle="--", label="Lesion boundary")
axis.set(xlabel="x (mm)", ylabel="y (mm)", aspect="equal", title="Grid viewed from above")
axis.legend()
plt.show()
print("Catheter placements:", len(placements))
print("Unique recovery points:", len(layout["recoveryPoints"]))


## 4. Prepare the two openCARP inputs

`prepareExperiment` follows four steps in `boundary_setup.py`:

1. Generate one regular slab using `mesher`, then write intact and lesion element tags on identical nodes.
2. Write the finite-electrode recovery coordinates.
3. Select the full-thickness left-edge stimulus strip. In the lesion condition, exclude every node touching a lesion element.
4. Write a complete parameter file for each condition.

The viable stimulus node sets differ between conditions, but the pacing schedule is identical. All fibres point along x. The function refuses to overwrite an incomplete preparation or reuse incompatible simulation settings.

This cell **runs the mesher**, but does not start openCARP.


In [ ]:
prepareExperiment(runFolder, config)

intactPositions = readNodePositions(runFolder / "mesh" / "intact")
lesionPositions = readNodePositions(runFolder / "mesh" / "lesion")
print("Nodes in each condition:", len(intactPositions))
print("Matching node positions:", np.array_equal(intactPositions, lesionPositions))
print((runFolder / "intact.par").read_text())


## 5. Run the intact and lesion conditions

This cell starts openCARP. The launch is a direct call to `openCARP +F <parameter file>`, preceded by `mpiexec -n <process count>` when using multiple processes. Read `simulateExperiment` to see that command.

Each completed condition contains transmembrane voltage (`vm.igb`) and recovered extracellular potentials (`phie_recovery.igb`). The code gathers each electrode's surface samples and averages them into `egm.npz`. It records completion only after both recordings and the electrode export are available.

Completed conditions are reused; incomplete output directories are not overwritten. Logs are `intact.log` and `lesion.log` in the run folder.


In [ ]:
simulateExperiment(runFolder, config, processCount=processCount)
checkCompletedSimulations(runFolder)


## 6. Check that the physiological experiment worked

Before measuring EFOV, check the last three beats. We require one activation per beat in the viable analysis region, complete repolarisation within the selected window, stable activation/amplitude/APD90 and propagation close to the intended planar +x wave. Activation in the lesion core also fails the check.

The fitted speed and direction are measured from activation times through the slab thickness. These are provisional pilot acceptance criteria, not established validation limits. Inspect failed criteria before changing the protocol.

Starting a new analysis archives previous derived results and figures. Raw simulation files are retained.


In [ ]:
startAnalysis(runFolder, config)
physiologyPassed = checkExperiment(runFolder, config)
makeFigures(runFolder, config)

display(pd.read_csv(runFolder / "physiologyChecks.csv"))
display(json.loads((runFolder / "physiologyStatus.json").read_text()))
display(Image(filename=str(runFolder / "figures" / "01_activation.png")))

if not physiologyPassed:
    raise RuntimeError("Physiology failed: inspect these outputs before analysing contributions")


## 7. Set one gain from the intact active reference

Use the final beat at the declared intact reference placement. The median amplitude of its nine selected bipolar recordings is scaled to 1.44 mV by default. Keep this **same gain** for every condition, placement and noise repeat.

This is imposed amplitude calibration. It does not independently validate the model's voltage scale.


In [ ]:
# Recheck the gate here so running later cells manually cannot accidentally bypass it.
if not physiologyPassed:
    raise RuntimeError("The active reference must pass the physiological checks")
gain = calibrateGain(runFolder, config)
display(json.loads((runFolder / "results" / "calibration.json").read_text()))


## 8. Build clean and noisy recordings

For each condition: apply the gain, add electrode noise, filter with the preceding history, and select the prespecified beat. Then build 16 unipoles, 9 maximum-orthogonal bipoles and 36 omnipolar recordings. Repeat zero is noise-free; repeats 1–3 are independent noise realisations by default.

`Shared_Pipeline/egm.py` contains these three recording calculations. Each recording has a label, trace and sixteen electrode weights. Those exact weights are saved, including any change in pair/direction caused by noise. We also save the noise-free signal at those same weights.

Bipolar distance uses the actual selected pair midpoint. Omnipolar distance uses the clique centroid. Individual electrode and alternative clique distances remain in the exports.


In [ ]:
if not physiologyPassed:
    raise RuntimeError("Inspect the failed physiology before continuing")
measurementTables = []
for condition in ["intact", "lesion"]:
    measurements = processCondition(runFolder, condition, config, gain)
    measurementTables.append(measurements)
    print(condition, "recordings exported:", len(measurements))

display(measurementTables[0][[
    "kind", "label", "noiseRepeat", "ppMv", "distanceViableMm", "selectedElectrodeIds"
]].head(10))


## 9. Estimate the contributing tissue

For the clean recordings at the selected catheter centres, reconstruct each waveform from the saved transmembrane voltage. `Shared_Pipeline/ground_truth.py` makes the approximation explicit:

- Estimate the voltage gradient on the regular 3D grid.
- Multiply by nodal intracellular conductivity and integration volume.
- Average point lead fields over each finite electrode and combine them using the saved recording weights.
- Add tissue in progressively larger full-thickness cylinders around the recording anchor.

Save two 80% radii: a persistent waveform-reconstruction radius and a cumulative node-energy radius. These answer different questions. The physical prefactor is omitted in this approximation; the next step checks waveform shape, signed correlation and a common scale against openCARP recovery before accepting a radius.

Noise repeats do not receive a copied clean FOV. They quantify measurement variation rather than adding biological models. This calculation can use substantial memory.


In [ ]:
contributionTables = []
for condition, measurements in zip(["intact", "lesion"], measurementTables):
    contributions = computeContributions(
        runFolder, condition, config, measurements, gain, physiologyPassed
    )
    contributionTables.append(contributions)
    print(condition, "contribution rows:", len(contributions))


## 10. Apply the acceptance criteria and make figures

`saveExperimentResults` checks reconstruction agreement, joins eligible contribution results to measurements, summarises the noise repeats and checks the far-region plateau and lesion transition. Failed or uncomputed contributions remain distinguishable in `points.csv`.

A completed pipeline is not automatically an accepted experiment. A passing pilot still requires mesh, time-step, electrode-quadrature and domain comparisons before choosing final parameters.


In [ ]:
measurements, pilotPassed = saveExperimentResults(
    runFolder, config, physiologyPassed, measurementTables, contributionTables
)
makeFigures(runFolder, config)

display(json.loads((runFolder / "results" / "pilotStatus.json").read_text()))
display(pd.read_csv(runFolder / "results" / "boundaryChecks.csv"))
for figureName in ["02_clean_voltage_distance.png", "03_contribution_radii.png",
                   "04_example_traces.png", "05_reconstruction_checks.png"]:
    display(Image(filename=str(runFolder / "figures" / figureName)))


## Where to go next

Read `physiologyChecks.csv` and `pilotStatus.json` first. Resolve capture/adaptation/window problems before drawing spatial conclusions. If the far region has not stabilised, enlarge the slab and the placement range. If reconstructed and recovered waveforms disagree, investigate that approximation before reporting contribution radii.

For reanalysis, retain the simulation settings and rerun from step 6 with your declared analysis changes. For a new mesh, pacing schedule, boundary or electrode layout, return to step 2 and choose a new folder. Do not change the configuration midway through analysis.

Start with the straight boundary. The geometry helper supports circles, but circular lesions need their own planarity/region assessment and a revised layout illustration.

`BOUNDARY_PILOT.md` contains terminal commands, output definitions and diagnostic processing instructions. `README.md` gives the source-code reading order.
